<a href="https://colab.research.google.com/github/jhughes7386/cosc-650-applied-llm-systems/blob/week-06/week-06/week6_advanced_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 6 (starter): Advanced Retrieval, Transform and Measure

Extends Week 5. The transformation shown here is pseudo-relevance feedback (PRF), which is fully local and needs no key; HyDE is guarded behind a key. Cells marked **TODO (you)** are yours. Dependency: `sentence-transformers`. For HyDE: `pip install openai` and a Gemini key.

## Part 1: Transform and re-rank


In [2]:
import os, re, pathlib, numpy as np
from collections import Counter

os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())

from sentence_transformers import SentenceTransformer, CrossEncoder, util
import faiss

embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Reuse the Week 5 Docker documentation corpus.
corpus = {
    'docker_overview': 'https://docs.docker.com/get-started/docker-overview/',
    'what_is_container': 'https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-a-container/',
    'what_is_image': 'https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-an-image/',
    'what_is_registry': 'https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-a-registry/',
    'what_is_compose': 'https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-docker-compose/',
    'image_layers': 'https://docs.docker.com/get-started/docker-concepts/building-images/understanding-image-layers/',
    'build_tag_publish': 'https://docs.docker.com/get-started/docker-concepts/building-images/build-tag-and-publish-an-image/',
    'running_containers': 'https://docs.docker.com/engine/containers/run/',
    'volumes': 'https://docs.docker.com/engine/storage/volumes/',
    'networking': 'https://docs.docker.com/engine/network/'
}

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Download and clean the URL as docs

In [3]:
import requests
from bs4 import BeautifulSoup

def fetch_doc(url):
    response = requests.get(url)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'html.parser')

    for tag in soup(['script', 'style', 'nav', 'footer']):
        tag.decompose()

    # Added: preserve separate HTML text blocks so paragraph chunking works.
    blocks = []

    for element in soup.find_all(['h1', 'h2', 'h3', 'h4', 'p', 'li']):
        text = re.sub(r'\s+', ' ', element.get_text(' ', strip=True)).strip()

        if text:
            blocks.append(text)

    # Added: separate each text block with a blank line for chunk_paragraph().
    return '\n\n'.join(blocks)

DOCS = {name: fetch_doc(url) for name, url in corpus.items()}

for name, text in DOCS.items():
    print(f'{name:25s} -> {len(text):,} characters')

docker_overview           -> 10,068 characters
what_is_container         -> 7,332 characters
what_is_image             -> 9,962 characters
what_is_registry          -> 6,598 characters
what_is_compose           -> 7,784 characters
image_layers              -> 10,494 characters
build_tag_publish         -> 8,301 characters
running_containers        -> 25,968 characters
volumes                   -> 18,730 characters
networking                -> 9,883 characters


Perform chunking

In [4]:
def chunk_fixed(text, size, overlap):

    text = re.sub(r'\s+', ' ', text).strip(); out, i = [], 0

    while i < len(text):

        out.append(text[i:i+size]); i += size - overlap

    return out

def chunk_paragraph(text):

    return [re.sub(r'\s+',' ',p).strip() for p in text.split('\n\n') if p.strip()]


# Added: apply each chunking configuration to all 10 documents.
configs = {
    'small (120/20)': [(name, chunk) for name, text in DOCS.items() for chunk in chunk_fixed(text, 120, 20)],
    'large (320/40)': [(name, chunk) for name, text in DOCS.items() for chunk in chunk_fixed(text, 320, 40)],
    'by-paragraph': [(name, chunk) for name, text in DOCS.items() for chunk in chunk_paragraph(text)]
}

for n, c in configs.items():

    print(f'{n:16s} -> {len(c)} chunks')


def build(chunks):

    # Added: embed only the text portion of each document chunk.
    e = embedder.encode([chunk for name, chunk in chunks], normalize_embeddings=True).astype('float32')

    idx = faiss.IndexFlatIP(e.shape[1]); idx.add(e); return idx


def retrieve(idx, chunks, q, k=3):

    qe = embedder.encode([q], normalize_embeddings=True).astype('float32')

    # FAISS uses -1 for missing results when fewer than k chunks exist.
    return [chunks[i] for i in idx.search(qe, k)[1][0] if i >= 0]


# Build FAISS indices
indices = {name: build(chunks) for name, chunks in configs.items()}

for name, index in indices.items():
    print(f'{name:16s} -> {index.ntotal} vectors')


small (120/20)   -> 1148 chunks
large (320/40)   -> 414 chunks
by-paragraph     -> 938 chunks
small (120/20)   -> 1148 vectors
large (320/40)   -> 414 vectors
by-paragraph     -> 938 vectors


In [5]:
STOP = set('a an the of to in on for and or is are how do does what so that with by as it why'.split())

def prf_expand(q, feedback_k=3, add=4):
    top = retrieve(
        indices['large (320/40)'],
        configs['large (320/40)'],
        q,
        k=feedback_k
    )

    words = re.findall(
        r'[a-zA-Z]+',
        ' '.join(chunk for name, chunk in top).lower()
    )

    query_words = set(re.findall(r'[a-zA-Z]+', q.lower()))

    cand = [
        w for w in words
        if w not in STOP and len(w) > 3 and w not in query_words
    ]

    return q + ' ' + ' '.join(
        w for w, _ in Counter(cand).most_common(add)
    )

In [7]:
# TODO (you): also add a re-ranking step and explain the regression (in Part 2).
def rerank(query, candidates, k=3):
    pairs = [(query, chunk) for name, chunk in candidates]
    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(scores, candidates),
        key=lambda x: x[0],
        reverse=True
    )

    return [candidate for score, candidate in ranked[:k]]

def retrieve_prf_reranked(q, candidate_k=10, final_k=3):
    transformed_q = prf_expand(q)

    candidates = retrieve(
        indices['large (320/40)'],
        configs['large (320/40)'],
        transformed_q,
        k=candidate_k
    )

    reranked = rerank(
        transformed_q,
        candidates,
        k=final_k
    )

    return transformed_q, reranked

## Part 2: Measure the effect

In [9]:
qa = [
    ('what is a Docker container', 'isolated processes'),
    ('what is a Docker image', 'read-only template'),
    ('what is a Docker registry', 'store and distribute images'),
    ('what is Docker Compose', 'multi-container applications'),
    ('what are Docker image layers', 'each layer'),
    ('how do you build tag and publish an image', 'docker push'),
    ('how do you run a Docker container', 'docker run'),
    ('what are Docker volumes used for', 'persist data'),
    ('what is Docker networking', 'network drivers'),
    ('what is Docker', 'open platform')
]

K = 3

def evaluate_retrieval(retrieved, chunks, answer_fact):
    relevant_retrieved = sum(
        1 for name, chunk in retrieved
        if answer_fact.lower() in chunk.lower()
    )

    total_relevant = sum(
        1 for name, chunk in chunks
        if answer_fact.lower() in chunk.lower()
    )

    precision = relevant_retrieved / K
    recall = relevant_retrieved / total_relevant if total_relevant else 0

    return precision, recall


results = []

for query, answer_fact in qa:
    baseline = retrieve(
        indices['large (320/40)'],
        configs['large (320/40)'],
        query,
        k=K
    )

    transformed_query, reranked = retrieve_prf_reranked(
        query,
        candidate_k=10,
        final_k=K
    )

    baseline_precision, baseline_recall = evaluate_retrieval(
        baseline,
        configs['large (320/40)'],
        answer_fact
    )

    reranked_precision, reranked_recall = evaluate_retrieval(
        reranked,
        configs['large (320/40)'],
        answer_fact
    )

    results.append({
        'query': query,
        'baseline_precision': baseline_precision,
        'reranked_precision': reranked_precision,
        'precision_delta': reranked_precision - baseline_precision,
        'baseline_recall': baseline_recall,
        'reranked_recall': reranked_recall,
        'recall_delta': reranked_recall - baseline_recall
    })

for result in results:
    print(f"\n{result['query']}")
    print(
        f"  Precision@3: "
        f"{result['baseline_precision']:.2f} -> "
        f"{result['reranked_precision']:.2f} "
        f"(Δ {result['precision_delta']:+.2f})"
    )
    print(
        f"  Recall@3:    "
        f"{result['baseline_recall']:.2f} -> "
        f"{result['reranked_recall']:.2f} "
        f"(Δ {result['recall_delta']:+.2f})"
    )


what is a Docker container
  Precision@3: 0.00 -> 0.00 (Δ +0.00)
  Recall@3:    0.00 -> 0.00 (Δ +0.00)

what is a Docker image
  Precision@3: 0.33 -> 0.00 (Δ -0.33)
  Recall@3:    1.00 -> 0.00 (Δ -1.00)

what is a Docker registry
  Precision@3: 0.00 -> 0.00 (Δ +0.00)
  Recall@3:    0.00 -> 0.00 (Δ +0.00)

what is Docker Compose
  Precision@3: 0.00 -> 0.00 (Δ +0.00)
  Recall@3:    0.00 -> 0.00 (Δ +0.00)

what are Docker image layers
  Precision@3: 0.00 -> 0.33 (Δ +0.33)
  Recall@3:    0.00 -> 0.17 (Δ +0.17)

how do you build tag and publish an image
  Precision@3: 0.33 -> 0.33 (Δ +0.00)
  Recall@3:    0.12 -> 0.12 (Δ +0.00)

how do you run a Docker container
  Precision@3: 0.67 -> 0.00 (Δ -0.67)
  Recall@3:    0.05 -> 0.00 (Δ -0.05)

what are Docker volumes used for
  Precision@3: 0.00 -> 0.00 (Δ +0.00)
  Recall@3:    0.00 -> 0.00 (Δ +0.00)

what is Docker networking
  Precision@3: 0.33 -> 0.00 (Δ -0.33)
  Recall@3:    0.50 -> 0.00 (Δ -0.50)

what is Docker
  Precision@3: 0.33 -> 0.00 

### Part 2: Analysis

**Queries that benefited:**

Only the Docker image layers improved, with precision increasing from 0.00 to 0.33 and recall from 0.00 to 0.17. This implies that the PRF expansion and reranking were able to move a relevant chunk into the final top 3 for this query.

**Queries that did not change:**

Container, registry, Compose, build/tag/publish, and volumes all had zero deltas. Since the retrieval metrics stayed the same, these queries did not benefit from the transformation and reranking. The build/tag/publish query is interesting because it already had relevant retrieval, but the new approach did not improve the results.

**Queries that worsened:**

Docker image, run a Docker container, networking, and Docker all had negative deltas. The largest regressions were seen in Docker image and Docker, where recall dropped from 1.00 to 0.00. "How do you run a Docker container" also dropped dramatically, with precision going from 0.67 to 0.00.

The results are surprising to me and hard to pinpoint. The PRF appears to be more sensitive to the terms it adds to a query because it is built from words in the initial retrieved chunks. These added terms do not always improve FAISS retrieval toward finding relevant chunks. When CrossEncoder is reranking, the relevant chunks could be ranked lower or may not have been included in the top candidates to begin with. Since these measurements combine both PRF and reranking, it is also unclear which step caused each regression.


## Part 3: Find one failure and explain it

In [10]:
failure_cases = [
    ('what is a Docker image', 'read-only template'),
    ('how do you run a Docker container', 'docker run'),
    ('what is Docker networking', 'network drivers'),
    ('what is Docker', 'open platform')
]

for failure_query, answer_fact in failure_cases:
    print('=' * 80)
    print(f'QUERY: {failure_query}')
    print(f'ANSWER FACT: {answer_fact}')

    # Baseline retrieval
    baseline = retrieve(
        indices['large (320/40)'],
        configs['large (320/40)'],
        failure_query,
        k=3
    )

    # PRF transformation
    transformed_query = prf_expand(failure_query)

    # Retrieve top 10 candidates using transformed query
    candidates = retrieve(
        indices['large (320/40)'],
        configs['large (320/40)'],
        transformed_query,
        k=10
    )

    # CrossEncoder reranking
    reranked = rerank(
        transformed_query,
        candidates,
        k=3
    )

    print('\nPRF transformed query:')
    print(transformed_query)

    print('\nBaseline top 3:')
    for i, (name, chunk) in enumerate(baseline, 1):
        relevant = answer_fact.lower() in chunk.lower()
        print(f'\n{i}. [{name}] relevant={relevant}')
        print(chunk)

    print('\nPRF top 10 candidates:')
    for i, (name, chunk) in enumerate(candidates, 1):
        relevant = answer_fact.lower() in chunk.lower()
        print(f'\n{i}. [{name}] relevant={relevant}')
        print(chunk)

    print('\nFinal CrossEncoder top 3:')
    for i, (name, chunk) in enumerate(reranked, 1):
        relevant = answer_fact.lower() in chunk.lower()
        print(f'\n{i}. [{name}] relevant={relevant}')
        print(chunk)

QUERY: what is a Docker image
ANSWER FACT: read-only template

PRF transformed query:
what is a Docker image namespace search some images

Baseline top 3:

1. [docker_overview] relevant=True
ief overview of some of those objects. Images An image is a read-only template with instructions for creating a Docker container. Often, an image is based on another image, with some additional customization. For example, you may build an image that is based on the Ubuntu image but includes the Apache web server and yo

2. [build_tag_publish] relevant=False
 the image, consisting of slash-separated components. For Docker Hub, the format follows [NAMESPACE/]REPOSITORY , where namespace is either a user's or organization's name. If no namespace is specified, library is used, which is the namespace for Docker Official Images. TAG : A custom, human-readable identifier that's 

3. [what_is_image] relevant=False
e image details page presents you with information regarding the layers of the image, the pac

### Part 3: Analysis

I chose to look at all four queries that worsened, looking at their top 10 candidates and final reranked results to see why the failures happen.

**PRF removed the relevant chunk:**

For "what is a Docker image," the baseline retrieved the relevant chunk as the #1 result, containing the "read-only template" answer fact. The PRF changed the query to "what is a Docker image namespace search some images." This did not include the relevant chunk in the transformed top 10 candidates. Since the CrossEncoder only reranks those candidates, it never could recover the relevant chunk. This caused precision to drop from 0.33 to 0.00 and recall from 1.00 to 0.00.


**Reranking pushed the relevant chunks out:**

For "how do you run a Docker container," the baseline had relevant chunks at #2 and #3. After PRF, relevant chunks were still present at #4, #7, and #8 in the top 10, but none made the final CrossEncoder top 3. This shows that while the relevant information was still available after PRF, the reranking step moved it out of the final results.

The same thing happened with "what is Docker." The relevant chunk was #1 in the baseline and was still in the transformed top 10 at #4. The final CrossEncoder top 3 contained three other Docker overview chunks instead, causing the relevant chunk to be removed.

**PRF weakened the ranking and reranking did not recover it:**

"what is Docker networking" shows a combination of the previous two issues. The relevant "network drivers" chunk was #1 in the baseline, but dropped to #5 with PRF. Despite this being available to the CrossEncoder, the final top 3 contained other networking chunks instead. **This means that PRF weakens the position of the relevant chunk, and reranking does not recover this.**

Overall, all 4 failures show that the PRF and reranking approach was too sensitive to the initial retrieval results. Some cases lost the relevant chunk completely, while others incorrectly ranked other related chunks higher in the CrossEncoder.


### HyDE (needs a key) and submit
HyDE generates a hypothetical answer and retrieves on it. With a key it slots in where PRF is; without one it is skipped.

In [ ]:
def hyde(q, model='gemini-2.5-flash-lite'):
    if not os.environ.get('GEMINI_API_KEY'):
        return '[API-BLOCKED] set GEMINI_API_KEY to generate a hypothetical answer, then embed and retrieve on it'
    from openai import OpenAI
    c=OpenAI(api_key=os.environ['GEMINI_API_KEY'], base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    return c.chat.completions.create(model=model, messages=[{'role':'user','content':'Write a one-sentence answer to: '+q}]).choices[0].message.content
print(hyde('how do indexes speed up reads'))
# Submit: PR with before/after deltas + a concrete cross-repo artifact with a classmate.
# Rubric: transform+rerank (30), before/after deltas (30), regression explained (20), cross-repo artifact (20).